In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Carpeta con los CSV generados por `transform`
CSV_DIR = Path("../data/csv")

# Puntos de la curva que se resumen en 7.7 (proporción del vídeo transcurrida)
MARCAS = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]

ret = pd.read_csv(CSV_DIR / "analytics_video_retention.csv")
perf = pd.read_csv(CSV_DIR / "video_performance.csv")

# --- 7.1 Estructura de la tabla ---------------------------------------------
print("== 7.1 Estructura ==")
print(f"Filas: {len(ret)} | vídeos distintos: {ret['video_id'].nunique()}")
print(f"Columnas: {list(ret.columns)}")

puntos_por_video = ret.groupby("video_id").size()
print("\nPuntos por vídeo (nº de puntos -> nº de vídeos):")
print(puntos_por_video.value_counts().sort_index().to_string())

grid = np.sort(ret["elapsedVideoTimeRatio"].unique())
print(f"\nValores distintos de elapsedVideoTimeRatio: {len(grid)}")
print(f"Mínimo: {grid.min()} | máximo: {grid.max()}")

# --- 7.2 Vídeos con Analytics pero sin curva de retención -------------------
con_analytics = perf[perf["views"].notna()]
sin_curva = con_analytics[~con_analytics["video_id"].isin(ret["video_id"])]

print(f"\n== 7.2 Vídeos con Analytics y sin curva: {len(sin_curva)} ==")
if not sin_curva.empty:
    cols = [
        "video_id", "title", "privacy_status", "format", "published_at",
        "duration_seconds", "views",
    ]
    print(sin_curva[cols].to_string(index=False))

# --- 7.3 Vídeos con curva que no están en video_performance -----------------
huerfanos = set(ret["video_id"]) - set(perf["video_id"])
print(f"\n== 7.3 Vídeos en retención ausentes de video_performance: {len(huerfanos)} (esperado: 0) ==")

# --- 7.4 Duplicados y nulos --------------------------------------------------
duplicados = int(ret.duplicated(["video_id", "elapsedVideoTimeRatio"]).sum())
print("\n== 7.4 Calidad ==")
print(f"Filas duplicadas (video_id, elapsedVideoTimeRatio): {duplicados} (esperado: 0)")
print("Nulos por columna:")
print(ret[["elapsedVideoTimeRatio", "audienceWatchRatio", "relativeRetentionPerformance"]]
      .isna().sum().to_string())

# --- 7.5 Rangos de valores ---------------------------------------------------
print("\n== 7.5 Rangos ==")
print(ret[["audienceWatchRatio", "relativeRetentionPerformance"]].describe().round(3).to_string())
print(f"\nPuntos con audienceWatchRatio > 1: {int((ret['audienceWatchRatio'] > 1).sum())}")

# --- 7.6 Curvas incompletas: ¿qué tienen en común? --------------------------
# Se considera incompleto un vídeo con menos puntos que la moda
moda = int(puntos_por_video.mode().iloc[0])
incompletos = puntos_por_video[puntos_por_video != moda]
print(f"\n== 7.6 Vídeos con un nº de puntos distinto de la moda ({moda}): {len(incompletos)} ==")
if not incompletos.empty:
    detalle = perf[perf["video_id"].isin(incompletos.index)][
        ["video_id", "title", "format", "duration_seconds", "views"]
    ].assign(puntos=lambda d: d["video_id"].map(incompletos))
    print(detalle.to_string(index=False))

# --- 7.7 Curva típica por formato (solo públicos con Analytics) -------------
publicos = perf[(perf["is_public"]) & (perf["views"].notna())][
    ["video_id", "format", "views", "averageViewPercentage"]
]
curva = ret.merge(publicos, on="video_id", how="inner")

# Punto de la rejilla más cercano a cada marca (evita depender de decimales exactos)
cercano = {m: float(grid[np.abs(grid - m).argmin()]) for m in MARCAS}
marcas = curva[curva["elapsedVideoTimeRatio"].isin(cercano.values())]
tabla = marcas.pivot_table(
    index="format",
    columns="elapsedVideoTimeRatio",
    values="audienceWatchRatio",
    aggfunc="median",
).round(3)

print("\n== 7.7 Mediana de audienceWatchRatio por formato en cada marca ==")
print(f"Marcas pedidas -> punto usado: {cercano}")
print(tabla.to_string())

# --- 7.8 Contraste con averageViewPercentage --------------------------------
# El promedio de la curva debería correlacionar con averageViewPercentage
resumen = (
    curva.groupby("video_id")
    .agg(
        media_curva=("audienceWatchRatio", "mean"),
        media_relativa=("relativeRetentionPerformance", "mean"),
        avg_view_pct=("averageViewPercentage", "first"),
        formato=("format", "first"),
    )
    .reset_index()
)
print("\n== 7.8 Correlación entre media de la curva y averageViewPercentage ==")
for formato, grupo in resumen.groupby("formato"):
    corr = grupo["media_curva"].corr(grupo["avg_view_pct"])
    print(f"{formato:6s} vídeos: {len(grupo):3d} | correlación de Pearson: {corr:.3f}")

== 7.1 Estructura ==
Filas: 37000 | vídeos distintos: 370
Columnas: ['elapsedVideoTimeRatio', 'audienceWatchRatio', 'relativeRetentionPerformance', 'report', 'scope', 'start_date', 'end_date', 'video_id']

Puntos por vídeo (nº de puntos -> nº de vídeos):
100    370

Valores distintos de elapsedVideoTimeRatio: 100
Mínimo: 0.01 | máximo: 1.0

== 7.2 Vídeos con Analytics y sin curva: 1 ==
   video_id                                                 title privacy_status format              published_at  duration_seconds  views
-gcVw6FnGxg ¡Aquí estamos todos, Betis está emitiendo en directo!       unlisted   live 2025-03-10T14:30:22+00:00                 0    0.0

== 7.3 Vídeos en retención ausentes de video_performance: 0 (esperado: 0) ==

== 7.4 Calidad ==
Filas duplicadas (video_id, elapsedVideoTimeRatio): 0 (esperado: 0)
Nulos por columna:
elapsedVideoTimeRatio           0
audienceWatchRatio              0
relativeRetentionPerformance    0

== 7.5 Rangos ==
       audienceWatchRatio  re